# 02 - Silver to Gold (Batch Layer)
Agregations metier + ecriture dans ClickHouse (base OLAP colonnaire)

Les resultats sont stockes dans **HDFS** (Gold) et dans **ClickHouse** pour le dashboard

In [1]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import clickhouse_connect

spark = SparkSession.builder \
    .appName("02_Silver_to_Gold") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.shuffle.partitions", "6") \
    .getOrCreate()

# Connexion ClickHouse
ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)

print("Spark connecte")
print("ClickHouse connecte")

Spark connecte
ClickHouse connecte


In [2]:
# Lire les donnees Silver
trips = spark.read.parquet("hdfs://namenode:9000/projet/silver/trips_clean/")
dates = spark.read.parquet("hdfs://namenode:9000/projet/silver/dates_clean/")
locations = spark.read.parquet("hdfs://namenode:9000/projet/silver/locations_clean/")
weather = spark.read.parquet("hdfs://namenode:9000/projet/silver/weather_clean/")

# Limiter a 1M lignes pour les performances sur machine locale
trips = trips.limit(1_000_000)

# Distribuer sur les 2 workers (6 partitions = 3 cores x 2 workers)
trips = trips.repartition(6)

print(f"Trips Silver: {trips.count()} lignes")
print(f"Partitions: {trips.rdd.getNumPartitions()}")

Trips Silver: 1000000 lignes
Partitions: 6


In [3]:
# Agregation 1 : Stats par borough
stats_borough = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("borough")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("total_amount"), 2).alias("avg_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(desc("nb_trips"))
)
stats_borough.show()

# Sauvegarder HDFS + ClickHouse
stats_borough.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_borough/")

pdf = stats_borough.toPandas()
ch_client.command("TRUNCATE TABLE batch_stats_borough")
ch_client.insert_df("batch_stats_borough", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_borough")

+-------------+--------+------------+--------+-------------+
|      borough|nb_trips|avg_distance|avg_fare|total_revenue|
+-------------+--------+------------+--------+-------------+
|    Manhattan|  657023|        5.15|   24.61|1.616826926E7|
|       Queens|  178085|        14.3|   72.05|1.283172979E7|
|     Brooklyn|  154310|        5.54|   25.14|   3878643.43|
|        Bronx|    8712|        4.46|   29.29|    255180.07|
|Staten Island|    1752|       27.85|   34.35|     60175.27|
|          EWR|     118|        4.51|   97.92|     11554.27|
+-------------+--------+------------+--------+-------------+

ClickHouse: 6 lignes inserees dans batch_stats_borough


In [4]:
# Agregation 2 : Stats par mois
stats_mensuel = (trips
    .join(broadcast(dates), "date_id")
    .groupBy("month", "year")
    .agg(
        count("*").alias("nb_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy("year", "month")
)
stats_mensuel.show(12)

# Sauvegarder HDFS + ClickHouse
stats_mensuel.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_par_mois/")

pdf = stats_mensuel.toPandas()
ch_client.command("TRUNCATE TABLE batch_stats_monthly")
ch_client.insert_df("batch_stats_monthly", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_monthly")

+-----+----+--------+-------------+------------+
|month|year|nb_trips|total_revenue|avg_distance|
+-----+----+--------+-------------+------------+
|    1|2024|   75949|   2468334.21|        5.37|
|    2|2024|   77146|    2437878.8|        4.48|
|    3|2024|   94784|   3053106.11|         5.4|
|    4|2024|   93878|   3057072.78|        8.86|
|    5|2024|   97840|   3268516.88|        8.18|
|    6|2024|   95450|   3151504.16|        8.79|
|    7|2024|   84404|   2873886.56|        6.67|
|    8|2024|   82982|   2849505.26|        6.35|
|    9|2024|  100063|   3413823.84|        5.83|
|   10|2024|  101688|   3483427.09|       10.15|
|   11|2024|   95816|    3148496.4|         4.5|
+-----+----+--------+-------------+------------+

ClickHouse: 11 lignes inserees dans batch_stats_monthly


In [5]:
# Agregation 3 : Top 10 zones
top_zones = (trips
    .join(broadcast(locations),
          trips.pickup_location_id == locations.location_id)
    .groupBy("zone", "borough")
    .agg(count("*").alias("nb_trips"))
    .orderBy(desc("nb_trips"))
    .limit(10)
)
top_zones.show(truncate=False)

# Sauvegarder HDFS + ClickHouse
top_zones.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/top_zones/")

pdf = top_zones.toPandas()
ch_client.command("TRUNCATE TABLE batch_top_zones")
ch_client.insert_df("batch_top_zones", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_top_zones")

+---------------------------------------------+---------+--------+
|zone                                         |borough  |nb_trips|
+---------------------------------------------+---------+--------+
|Jamaica                                      |Queens   |143352  |
|East Chelsea                                 |Manhattan|93167   |
|Clinton East                                 |Manhattan|85918   |
|East Village                                 |Manhattan|79039   |
|Governor's Island/Ellis Island/Liberty Island|Manhattan|68591   |
|Flatiron                                     |Manhattan|54828   |
|Central Park                                 |Manhattan|53921   |
|Garment District                             |Manhattan|52829   |
|Greenpoint                                   |Brooklyn |49855   |
|Green-Wood Cemetery                          |Brooklyn |48620   |
+---------------------------------------------+---------+--------+

ClickHouse: 10 lignes inserees dans batch_top_zones


In [6]:
# Agregation 4 : Impact meteo
stats_meteo = (trips
    .filter(col("weather_id") != -1)
    .join(broadcast(weather), "weather_id")
    .groupBy("weather_id", "temp_max", "precipitation", "snow")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("total_amount"), 2).alias("avg_fare")
    )
)
stats_meteo.show()

# Sauvegarder HDFS + ClickHouse
stats_meteo.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/stats_meteo/")

pdf = stats_meteo.toPandas()
pdf['weather_id'] = pdf['weather_id'].astype(int)
ch_client.command("TRUNCATE TABLE batch_stats_weather")
ch_client.insert_df("batch_stats_weather", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_stats_weather")

+----------+--------+-------------+----+--------+------------+--------+
|weather_id|temp_max|precipitation|snow|nb_trips|avg_distance|avg_fare|
+----------+--------+-------------+----+--------+------------+--------+
|        33|   27.37|          0.0| 0.0|    3865|         4.4|   32.53|
|        23|    33.7|         0.53| 0.0|    3100|        4.52|   33.24|
|        18|   28.45|          0.0| 0.0|    3204|        5.06|    34.1|
|        24|   34.26|         6.13| 0.0|    3023|        4.61|   33.26|
|        22|   28.26|          0.0| 0.0|    2107|        4.89|   33.62|
|        30|   21.45|         0.02| 0.0|    2846|        4.67|   33.71|
|        29|   24.69|          0.0| 0.0|    2753|        5.72|   38.54|
|        17|   26.89|          0.0| 0.0|    3978|       18.03|   30.34|
|        10|   14.01|         0.21| 0.0|    2293|        5.43|   36.12|
|         6|    6.12|         0.95| 0.0|    3216|        3.42|   27.24|
|        21|   27.74|          0.0| 0.0|    2326|        4.84|  

In [7]:
# Agregation 5 : Matrice Origine-Destination (Top 20)
loc_pu = locations.withColumnRenamed("zone", "zone_pickup") \
    .withColumnRenamed("location_id", "pickup_location_id") \
    .select("pickup_location_id", "zone_pickup")

loc_do = locations.withColumnRenamed("zone", "zone_dropoff") \
    .withColumnRenamed("location_id", "dropoff_location_id") \
    .select("dropoff_location_id", "zone_dropoff")

od_matrix = (trips
    .join(broadcast(loc_pu), on="pickup_location_id")
    .join(broadcast(loc_do), on="dropoff_location_id")
    .groupBy("zone_pickup", "zone_dropoff")
    .agg(
        count("*").alias("nb_trips"),
        round(avg("total_amount"), 2).alias("avg_fare")
    )
    .orderBy(desc("nb_trips"))
    .limit(20)
)
od_matrix.show(truncate=False)

# Sauvegarder HDFS + ClickHouse
od_matrix.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/gold/od_matrix/")

pdf = od_matrix.toPandas()
ch_client.command("TRUNCATE TABLE batch_od_matrix")
ch_client.insert_df("batch_od_matrix", pdf)
print(f"ClickHouse: {len(pdf)} lignes inserees dans batch_od_matrix")

print("\nSilver -> Gold TERMINE (HDFS + ClickHouse)")

+---------------------------------------------+---------------------------------------------+--------+--------+
|zone_pickup                                  |zone_dropoff                                 |nb_trips|avg_fare|
+---------------------------------------------+---------------------------------------------+--------+--------+
|Jamaica                                      |Sunset Park West                             |7177    |94.48   |
|Clinton East                                 |East Chelsea                                 |5975    |15.99   |
|Governor's Island/Ellis Island/Liberty Island|Mott Haven/Port Morris                       |5641    |15.55   |
|East Chelsea                                 |Clinton East                                 |4803    |19.15   |
|East Chelsea                                 |Sunset Park West                             |4449    |22.54   |
|Clinton East                                 |Clinton East                                 |4341    |15

In [8]:
ch_client.close()
spark.stop()
print("Connexions fermees")

Connexions fermees
